# Imports

In [89]:
import pandas as pd
import numpy as np
import sklearn

# Data reading

In [90]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

# EDA

## Feature Records

1. Numerical Columns
- LotFrontage
- LotArea
- YearBuilt
- YearRemodAdd
- MasVnrArea
- BsmtFinSF1
- BsmtFinSF2
- BsmtUnfSF
- TotalBsmtSF
- 1stFlrSF
- 2ndFlrSF
- LowQualFinSF
- GrLivArea
- BsmtFullBath
- BsmtHalfBath
- FullBath
- HalfBath
- Bedroom
- Kitchen
- TotRmsAbvGrd
- Fireplaces
- GarageYrBlt
- GarageArea
- GarageCars
- WoodDeckSF
- OpenPorchSF
- EnclosedPorch
- 3SsnPorch
- ScreenPorch
- PoolArea
- MiscVal
- MoSold
- YrSold

2. Nominal Columns
- MSZoning
- LotConfig
- Neighborhood
- Condition1
- Condition2
- RoofStyle
- RoofMatl
- Exterior1st
- Exterior2nd
- MasVnrType
- Foundation
- Heating
- MiscFeature

3. Ordinal Columns
- MSSubClass
- LotShape
- Street
- Alley
- LandContour
- Utilities
- LandSlope
- Bldgtype
- HouseStyle
- OverallQual
- OverallCond
- ExterQual
- ExterCond
- BsmtQual
- BsmtCond
- BsmtExposure
- BsmtFinType1
- BsmtFinType2
- HeatingQC
- CentralAir
- Electrical
- KitchenQual
- Functional
- FireplaceQu
- GarageType
- GarageFinish
- GarageQual
- GarageCond
- PavedDrive
- PoolQC
- Fence
- SaleType
- SaleCondition

In [91]:
train_df = train_df.drop(columns="Id")
test_df = test_df.drop(columns="Id")

In [92]:
numeric_columns = set([
    "LotFrontage",
    "LotArea",
    "YearBuilt",
    "YearRemodAdd",
    "MasVnrArea",
    "BsmtFinSF1",
    "BsmtFinSF2",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "LowQualFinSF",
    "GrLivArea",
    "BsmtFullBath",
    "BsmtHalfBath",
    "FullBath",
    "HalfBath",
    "Bedroom",
    "Kitchen",
    "TotRmsAbvGrd",
    "Fireplaces",
    "GarageYrBlt",
    "GarageArea",
    "GarageCars",
    "WoodDeckSF",
    "OpenPorchSF",
    "EnclosedPorch",
    "3SsnPorch",
    "ScreenPorch",
    "PoolArea",
    "MiscVal",
    "MoSold",
    "YrSold",
])

In [93]:
nominal_columns = set([
    "MSZoning",
    "LotConfig",
    "Neighborhood",
    "Condition1",
    "Condition2",
    "RoofStyle",
    "RoofMatl",
    "Exterior1st",
    "Exterior2nd",
    "MasVnrType",
    "Foundation",
    "Heating",
    "MiscFeature",
])

In [94]:
ordinal_columns = set([
    "MSSubClass",
    "LotShape",
    "Street",
    "Alley",
    "LandContour",
    "Utilities",
    "LandSlope",
    "Bldgtype",
    "HouseStyle",
    "OverallQual",
    "OverallCond",
    "ExterQual",
    "ExterCond",
    "BsmtQual",
    "BsmtCond",
    "BsmtExposure",
    "BsmtFinType1",
    "BsmtFinType2",
    "HeatingQC",
    "CentralAir",
    "Electrical",
    "KitchenQual",
    "Functional",
    "FireplaceQu",
    "GarageType",
    "GarageFinish",
    "GarageQual",
    "GarageCond",
    "PavedDrive",
    "PoolQC",
    "Fence",
    "SaleType",
    "SaleCondition",
])

## Missing Data

In [95]:
def clear_sparse(df):
    l = df.columns[df.isna().any()].tolist()
    k = []
    for i in l:
        if 100 * df[i].isna().sum() / len(df[i]) >= 30:
            k.append(i)
    df = df.drop(columns=k)

    missing_cols = list(set(l).difference(set(k)))

    return df, missing_cols

In [96]:
def feature_split(cols):
    numeric = set(cols).intersection(numeric_columns)
    nominal = set(cols).intersection(nominal_columns)
    ordinal = set(cols).intersection(ordinal_columns)

    return numeric, nominal, ordinal

In [97]:
from sklearn.impute import SimpleImputer
def median_impute(df, cols):
    imputer = SimpleImputer(strategy="median")
    imputer.fit(df[cols])
    vals = imputer.transform(df[cols])

    return df, imputer

In [98]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
def knn_impute(df, cols):
    imputer = KNNImputer(n_neighbors=3, weights="uniform")

    scaler = StandardScaler()
    scaler.fit(df)

    df_impute = imputer.fit_transform(scaler.transform(df))

    df = pd.DataFrame(scaler.inverse_transform(df_impute), df.columns)

    return df

In [99]:
train_df, train_miss = clear_sparse(train_df)
test_df, test_miss = clear_sparse(test_df)

In [100]:
train_numeric_miss, train_nominal_miss, train_ordinal_miss = feature_split(train_miss)
test_numeric_miss, test_nominal_miss, test_ordinal_miss = feature_split(test_miss)

In [101]:
train_df, train_numeric_imputer = median_impute(train_df, list(train_numeric_miss))
test_df, test_numeric_imputer = median_impute(test_df, list(test_numeric_miss))

In [ ]:
train_df, train_ordinal_imputer = knn_impute(train_df, list(train_ordinal_miss))
test_df, test_ordinal_imputer = knn_impute(test_df, list(test_ordinal_miss))